In [1]:
import sqlite3
import os

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
    SpanExporter,
    SpanExportResult,
)
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter
)

from starter import rag
from rag_helper import RAGBase


# =====================================================
# SQLite Span Exporter
# =====================================================

class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)

        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)

        self.conn.commit()


    def export(self, spans):

        for span in spans:
            attrs = dict(span.attributes or {})

            self.conn.execute(
                """
                INSERT INTO spans 
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                )
            )

        self.conn.commit()

        return SpanExportResult.SUCCESS


    def shutdown(self):
        self.conn.close()


    def force_flush(self):
        return True



# =====================================================
# Configure OpenTelemetry
# =====================================================

if os.path.exists("traces.db"):
    os.remove("traces.db")


provider = TracerProvider()


# Memory exporter for reading spans in Python
memory_exporter = InMemorySpanExporter()


provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)


provider.add_span_processor(
    SimpleSpanProcessor(memory_exporter)
)


provider.add_span_processor(
    SimpleSpanProcessor(
        SQLiteSpanExporter("traces.db")
    )
)


trace.set_tracer_provider(provider)


tracer = trace.get_tracer("llm-zoomcamp")



# =====================================================
# RAG with tracing
# =====================================================

class RAGTraced(RAGBase):


    def search(self, query, num_results=5):

        with tracer.start_as_current_span("search") as span:

            results = super().search(
                query,
                num_results=num_results
            )

            span.set_attribute(
                "num_results",
                len(results)
            )

            return results



    def llm(self, prompt):

        with tracer.start_as_current_span("llm") as span:

            response = super().llm(prompt)

            usage = response.usage

            if usage:

                input_tokens = usage.input_tokens or 0
                output_tokens = usage.output_tokens or 0


                # your pricing
                cost = (
                    input_tokens * 0.10 +
                    output_tokens * 0.40
                ) / 1_000_000


                span.set_attribute(
                    "input_tokens",
                    input_tokens
                )

                span.set_attribute(
                    "output_tokens",
                    output_tokens
                )

                span.set_attribute(
                    "cost",
                    cost
                )


                print(
                    f"""
Token usage:
input={input_tokens}
output={output_tokens}
cost=${cost:.6f}
"""
                )


            return response



    def rag(self, query):

        with tracer.start_as_current_span("rag") as span:

            span.set_attribute(
                "query",
                query
            )


            search_results = self.search(query)


            prompt = self.build_prompt(
                query,
                search_results
            )


            response = self.llm(prompt)


            return response



# =====================================================
# Create traced RAG instance
# =====================================================

rag_traced = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    model=rag.model,
)



# =====================================================
# Run query
# =====================================================

q1 = "How does the agentic loop keep calling the model until it stops?"


answer = rag_traced.rag(q1)


print("\nANSWER:")
print(answer)



# =====================================================
# Read spans and calculate duration
# =====================================================

print("\nSPAN DURATIONS:")


spans = memory_exporter.get_finished_spans()


print(
    f"Number of spans: {len(spans)}"
)


for span in spans:

    duration_ms = (
        span.end_time -
        span.start_time
    ) / 1_000_000


    print(
        f"{span.name}: {duration_ms:.3f} ms"
    )



# =====================================================
# Read SQLite data
# =====================================================

print("\nSQLITE RESULTS:")


conn = sqlite3.connect("traces.db")


cursor = conn.execute(
    """
    SELECT
        name,
        input_tokens,
        output_tokens,
        cost,
        (end_time - start_time) / 1000000 AS duration_ms
    FROM spans
    """
)


for row in cursor.fetchall():

    print(row)


conn.close()

{
    "name": "search",
    "context": {
        "trace_id": "0x7977358ff6701b38c97c3ea59fbcba2a",
        "span_id": "0x029b11c236f53345",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xde81801fafc8ab07",
    "start_time": "2026-07-20T16:20:32.164452Z",
    "end_time": "2026-07-20T16:20:32.166098Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "02cfec68-8387-46aa-8f9c-a2ae7534f68c",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

Token usage:
input=7111
output=108
cost=$0.000754

{
    "name": "llm",
    "context": {
        "trace_id": "0x7977358ff6701b38c97c3ea59fbcba2a",
        "span_i

Question 1 How many spans does the trace produce?

Number of spans: 3

Answer: 3

Question 2 How many input tokens do we see for the LLM call? 

Token usage:
input=7111
output=108
cost=$0.000754

Answer: 7000

Question 3 For a typical query, roughly how long does the LLM call take? (1 point)

SPAN DURATIONS:

search: 1.646 ms
llm: 2119.928 ms
rag: 2132.547 ms

Answer: Over 2000ms

Question 4: Which span names appear in the spans table? 

SQLITE RESULTS:
('search', None, None, None, 1)
('llm', 7111, 108, 0.0007543000000000001, 2119)
('rag', None, None, None, 2132)

Answer: rag, search, and llm

Question 5: Excluding the rag span, which span type takes the most total time?
Answer: llm

Qusetion 6: How much do the input tokens vary across 4 runs of the same query? (1 point)

Answer: They are identical